# NB2 — Estimation (Abrigo Rev-2 migration)

**Purpose.** This notebook executes the Rev-2 mean-β primary OLS+HAC(4) regression and the 14-row sensitivity ladder; specification tests live in NB3 (`03_tests_and_sensitivity.ipynb`).

**Upstream input.** NB1 panel-fingerprint JSON at `notebooks/abrigo_y3_x_d/estimates/panel_fingerprint.json` (the byte-exact upstream contract emitted by NB1 §1) and the 14 Phase 5a panel parquets at `contracts/.scratch/2026-04-25-task110-rev2-data/` (`panel_row_01_primary.parquet` … `panel_row_14_wc_cpi_weights_sens.parquet`) plus the published Rev-5.3.2 gate verdict at `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json`.

**Downstream consumers.** NB3 consumes the per-row estimates JSON emitted by NB2 §1–§6 (point-coefficient + HAC(4) SE record per spec-row + sensitivity-row); the 14-row resolution table renders into the auto-rendered README via the Jinja2 template.

**Pre-committed gate verdict.** FAIL (one-sided T3b on β̂_X_d). Reproduced byte-exact from `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json`; never re-estimated in this migration. Anti-fishing invariants `N_MIN=75`, `POWER_MIN=0.80`, `MDES_SD=0.40` and `MDES_FORMULATION_HASH=4940360dcd2987…cefa` are immutable through Rev-5.3.x.

---

## §0 — Notebook header + panel-load sanity



### Why-markdown (4-part citation block)

**Reference.**

- Rev-2 spec, autonomous track A, at `contracts/.scratch/2026-04-25-task110-rev2-spec-A-autonomous.md` (655 lines; §3 functional form / OLS+HAC(4) Newey–West; §6 T3b 90% one-sided gate definition; §11.A convex-payoff insufficiency caveat — mean-β is necessary-but-insufficient for option-like product pricing).
- NB1 panel-fingerprint manifest at `notebooks/abrigo_y3_x_d/estimates/panel_fingerprint.json` (the byte-exact upstream contract emitted by NB1 §1; consumed in NB2 §1+ to drive the 14-row resolution ladder).
- Phase 5b published gate verdict at `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json` (SHA-256 emitted by NB1 §0; `gate_verdict = "FAIL"`; `β̂_X_d = -2.7987e-8`; `HAC(4) SE = 1.4234e-8`; `t-stat = -1.9662`; `n = 76`; `window = [2024-09-27, 2026-03-13]`).
- Rev-5.3.5 β-disposition memo at `contracts/.scratch/2026-04-26-mr-beta-1-1-halt-resolution-beta.md` (the disposition that reclassifies the X_d source address `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606` as Minteo-fintech, OUT of Mento-native scope; Rev-2 closes scope-mismatch close-out on the Minteo-fintech X_d, not the Mento-native COPm hedge-demand surface at `0x8A567e2aE79CA692Bd748aB832081C45de4041eA`).
- Phase 5a Data Engineer outputs at `contracts/.scratch/2026-04-25-task110-rev2-data/` (`data_dictionary.md` is the canonical column-name authority; `manifest.md` is the row-summary; `_audit_summary.json` is the machine-readable per-row audit consumed by NB1 §0).
- Y₃ inequality-differential design at `contracts/docs/superpowers/specs/2026-04-24-y3-inequality-differential-design.md` (4-country panel CO/BR/KE/EU; equal-weight aggregation; pre-registered WC-CPI weights 60/25/15).
- X_d carbon-basket design at `contracts/docs/superpowers/specs/2026-04-24-carbon-basket-xd-design.md` (carbon-basket user-volume series; partition rule `trader != BancorArbitrage` per `project_carbon_user_arb_partition_rule`).
- Project memory: `project_mdes_formulation_pin` (Cohen f² formulation hash `4940360dcd2987…cefa`; free-tuning MDES_SD upward is anti-fishing-banned); `project_y3_inequality_differential_design`; `project_abrigo_inequality_hedge_thesis`; `project_abrigo_convex_instruments_inequality`; `project_abrigo_mento_native_only` (β-corrigendum: Rev-2's Minteo-fintech X_d close-out triggers the Minteo-fintech scope-mismatch dispatch to Task 11.P β-track on the Mento-native COPm surface).

**Why used.** NB2 is the estimation layer of the Rev-2 migration. NB1 produced the panel-fingerprint manifest that pinned the byte-exact upstream contract (14 panels, row counts, methodology tags, date windows, gate-verdict SHA-256). NB2 §0 establishes the panel-load contract that all downstream NB2 §1–§6 estimation cells will consume: byte-exact load of `panel_row_01_primary.parquet`, deterministic seed pin via `env.pin_seed`, schema validation against the Phase 5a `data_dictionary.md`. The Rev-2 published estimates (β̂_X_d = −2.7987e−8, HAC(4) SE = 1.4234e−8, t-stat = −1.9662, n = 76) are byte-exact-immutable per Rev-5.3.x anti-fishing invariants — NB2's job is to **reproduce** them, not re-estimate. Per `feedback_notebook_citation_block`, this 4-part block is non-negotiable for every estimation / sensitivity decision; the trio-checkpoint discipline (`feedback_notebook_trio_checkpoint`) HALTs after the §0 trio so sub-task 9 (NB2 §1 primary OLS+HAC(4)) dispatches separately.

**Relevance to results.** Byte-exact panel-load with seed pin is the necessary condition for byte-exact regression-coefficient reproduction in NB2 §1. The Rev-2 published primary-row estimate is computed on a panel of n = 76 Friday-anchored weeks over `[2024-09-27, 2026-03-13]` with column set `(week_start, y3_value, copm_diff, brl_diff, kes_diff, eur_diff, x_d, vix_avg, oil_return, us_cpi_surprise, banrep_rate_surprise, fed_funds_weekly, intervention_dummy)` — every dimension matching the Phase 5a `data_dictionary.md` byte-for-byte. Any drift surfaces here as a panel-construction or seeding bug rather than propagating silently into the §1 OLS coefficient. The `pin_seed(seed=42)` call is invariant across NB1 / NB2 / NB3; the `PYTHONHASHSEED` propagates to nbconvert child processes only (per the `env.py` docstring caveat), but bootstrap RNG state in NB3 is what actually consumes the seed — NB2 §0 / §1 are deterministic by closed-form OLS + HAC(4), not seed-dependent.

**Connection to product.** NB2 is the calibration-layer notebook for the Abrigo simulator: the 14-row mean-β resolution ladder produces the linear-hedge calibration inputs (point-β, HAC(4) SE, n, gate decision per row) that the simulator consumes for first-stage payoff matching. Per Rev-2 spec §11.A, mean-β is necessary-but-insufficient for the convex (option-like) instruments Abrigo sells against macroeconomic shocks viewed through the inequality lens (`project_abrigo_convex_instruments_inequality`); Rev-3 ζ-group (quantile / GARCH-X / lower-tail / option-implied vol) is where convex-payoff fitness gets tested, deferred to Task 11.O.ζ-α per `project_compact_survival_2026_04_26`. The Rev-2 migration closes scope-mismatch close-out under Rev-5.3.5: the Minteo-fintech X_d at `0xc92e8fc2…` is reclassified out of Mento-native scope, and the canonical Mento-native COPm at `0x8A567e2a…` is the actual hedge-demand surface — its own panel + estimation will use this same notebook scaffolding pattern under Task 11.P.spec-β / Task 11.P.exec-β (β-track).



In [1]:
"""NB2 §0 — panel-load sanity (primary-row byte-exact contract).

Loads `panel_row_01_primary.parquet` from the Phase 5a Data Engineer output
directory at `contracts/.scratch/2026-04-25-task110-rev2-data/`, pins the
deterministic seed via `env.pin_seed(seed=42)`, and asserts byte-exact match
against the Phase 5a `data_dictionary.md` schema (n = 76 rows, Friday-anchored
window `[2024-09-27, 2026-03-13]`, 13 canonical columns). The estimation cells
in NB2 §1+ consume this same panel; this trio establishes the load contract
that downstream cells inherit.

Functional-Python style: frozen dataclasses, free pure functions, full typing,
no inheritance.
"""
from __future__ import annotations

import importlib.util
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Final

import duckdb
import pandas as pd

# ---- Locate env.py and load by file path (notebooks/abrigo_y3_x_d/ is not on sys.path) ----

def _locate_abrigo_dir() -> Path:
    """Find the abrigo_y3_x_d/ directory that holds env.py."""
    cwd = Path.cwd().resolve()
    if (cwd / "env.py").is_file():
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / "contracts" / "notebooks" / "abrigo_y3_x_d"
        if (candidate / "env.py").is_file():
            return candidate
        candidate2 = parent / "notebooks" / "abrigo_y3_x_d"
        if (candidate2 / "env.py").is_file():
            return candidate2
    raise RuntimeError(f"Could not locate abrigo_y3_x_d/env.py starting from cwd={cwd}")


_ABRIGO_DIR: Final[Path] = _locate_abrigo_dir()
_ENV_PATH: Final[Path] = _ABRIGO_DIR / "env.py"
_spec = importlib.util.spec_from_file_location("abrigo_env", _ENV_PATH)
env = importlib.util.module_from_spec(_spec)
sys.modules["abrigo_env"] = env
_spec.loader.exec_module(env)

# ---- Pre-committed Phase 5a primary-row schema (binding upstream contract) ----

PRIMARY_PARQUET_NAME: Final[str] = "panel_row_01_primary.parquet"
PRIMARY_METHODOLOGY: Final[str] = "y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable"
PRIMARY_WINDOW: Final[tuple[str, str]] = ("2024-09-27", "2026-03-13")
PRIMARY_N: Final[int] = 76

# Canonical column set per Phase 5a `data_dictionary.md` §1–§4 (order-insensitive).
EXPECTED_COLUMNS: Final[frozenset[str]] = frozenset({
    # Index
    "week_start",
    # Outcome (LHS) + per-country diagnostics
    "y3_value", "copm_diff", "brl_diff", "kes_diff", "eur_diff",
    # Variable of interest (X_d)
    "x_d",
    # Control set γ_1 … γ_6
    "vix_avg", "oil_return", "us_cpi_surprise",
    "banrep_rate_surprise", "fed_funds_weekly", "intervention_dummy",
})

# Deterministic seed for any RNG-dependent downstream cell (NB3 bootstrap).
# Mirror the value used by NB1 trio cells; closed-form OLS + HAC(4) in NB2 §1
# is not seed-dependent, but pinning here matches the
# `feedback_notebook_citation_block` invariant and is required by sub-plan §C-8.
SEED: Final[int] = 42


@dataclass(frozen=True, slots=True)
class PanelLoadContract:
    """Sanity-check fingerprint of the loaded primary panel.

    Emitted byte-exact alongside the panel DataFrame so downstream cells in
    NB2 §1+ can assert continuity.
    """
    parquet_path: Path
    n_rows: int
    dt_min: str
    dt_max: str
    columns: tuple[str, ...]
    source_methodology: str


# ---- Pure functions (no inheritance, no globals mutated) ----

def _phase5a_data_dir() -> Path:
    return env._CONTRACTS_DIR / ".scratch" / "2026-04-25-task110-rev2-data"


def _read_primary_panel(parquet_path: Path) -> pd.DataFrame:
    """Read the primary parquet via DuckDB (read-only, no row mutation)."""
    if not parquet_path.is_file():
        raise FileNotFoundError(f"Phase 5a primary parquet missing: {parquet_path}")
    conn = duckdb.connect()
    try:
        return conn.execute(f"SELECT * FROM read_parquet('{parquet_path}')").fetchdf()
    finally:
        conn.close()


def _query_methodology(parquet_path: Path) -> str:
    """Resolve the source-methodology literal for the primary panel from the
    Phase 5a `_audit_summary.json` (the panel parquet itself omits the
    methodology column per the data-dictionary §6 metadata-not-on-row rule).
    """
    import json
    audit_path = parquet_path.parent / "_audit_summary.json"
    with audit_path.open() as fh:
        audit = json.load(fh)
    return audit["row_01_primary"]["y3_methodology"]


def build_panel_contract(data_dir: Path) -> tuple[pd.DataFrame, PanelLoadContract]:
    """Load the primary panel and emit its byte-exact fingerprint."""
    parquet_path = data_dir / PRIMARY_PARQUET_NAME
    df = _read_primary_panel(parquet_path)
    methodology = _query_methodology(parquet_path)
    dt_min = df["week_start"].min().strftime("%Y-%m-%d")
    dt_max = df["week_start"].max().strftime("%Y-%m-%d")
    contract = PanelLoadContract(
        parquet_path=parquet_path,
        n_rows=int(len(df)),
        dt_min=dt_min,
        dt_max=dt_max,
        columns=tuple(df.columns),
        source_methodology=methodology,
    )
    return df, contract


# ---- Execute the panel-load + seed-pin sanity check ----

# Pin deterministic RNG state. PYTHONHASHSEED applies to child processes only
# (env.py docstring caveat); numpy + Python random are pinned in this process.
env.pin_seed(seed=SEED)

_DATA_DIR = _phase5a_data_dir()
panel_primary, contract = build_panel_contract(_DATA_DIR)

# Byte-exact assertions vs. Phase 5a data_dictionary.md (the load-bearing contract)
assert contract.n_rows == PRIMARY_N, (
    f"Primary panel n_rows drift: got {contract.n_rows}, expected {PRIMARY_N}"
)
assert (contract.dt_min, contract.dt_max) == PRIMARY_WINDOW, (
    f"Primary window drift: got ({contract.dt_min}, {contract.dt_max}), "
    f"expected {PRIMARY_WINDOW}"
)
_actual_cols = frozenset(contract.columns)
assert _actual_cols == EXPECTED_COLUMNS, (
    f"Primary panel column-set drift:\n"
    f"  missing = {EXPECTED_COLUMNS - _actual_cols}\n"
    f"  extra   = {_actual_cols - EXPECTED_COLUMNS}"
)
assert contract.source_methodology == PRIMARY_METHODOLOGY, (
    f"Primary methodology drift: got {contract.source_methodology!r}, "
    f"expected {PRIMARY_METHODOLOGY!r}"
)

# Friday-anchored invariant per data_dictionary.md §1.1
_weekday_set = set(panel_primary["week_start"].dt.weekday.unique())
assert _weekday_set == {4}, (
    f"Friday-anchor invariant violated: weekdays observed = {sorted(_weekday_set)} "
    f"(expected {{4}} = Friday)"
)

# Emit panel-load summary (sanity print; downstream JSON emission lives in NB2 §1+)
print(f"Phase 5a data dir : {_DATA_DIR}")
print(f"Primary parquet   : {contract.parquet_path.name}")
print(f"n_rows            : {contract.n_rows}")
print(f"window            : [{contract.dt_min}, {contract.dt_max}]")
print(f"columns ({len(contract.columns)})    : {list(contract.columns)}")
print(f"source_methodology: {contract.source_methodology}")
print(f"seed (env.pin_seed): {SEED}")
print()
print("Panel head (first 2 rows):")
print(panel_primary.head(2).to_string(index=False))


Phase 5a data dir : /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/.scratch/2026-04-25-task110-rev2-data
Primary parquet   : panel_row_01_primary.parquet
n_rows            : 76
window            : [2024-09-27, 2026-03-13]
columns (13)    : ['week_start', 'y3_value', 'copm_diff', 'brl_diff', 'kes_diff', 'eur_diff', 'x_d', 'vix_avg', 'oil_return', 'us_cpi_surprise', 'banrep_rate_surprise', 'fed_funds_weekly', 'intervention_dummy']
source_methodology: y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable
seed (env.pin_seed): 42

Panel head (first 2 rows):
week_start  y3_value  copm_diff  brl_diff  kes_diff  eur_diff          x_d  vix_avg  oil_return  us_cpi_surprise  banrep_rate_surprise  fed_funds_weekly  intervention_dummy
2024-09-27  0.014237   0.003570  0.012624       NaN  0.026519  4604.160458   15.804   -0.056576              0.0                 0.000              4.83                   1
2024-10-04 -0.005903  -0.004091 -0.001508       

### Interpretation

**Panel loaded byte-exact against the Phase 5a contract.** `panel_row_01_primary.parquet` reads as n = 76 Friday-anchored rows over `[2024-09-27, 2026-03-13]` with the 13 canonical columns enumerated in `data_dictionary.md` (`week_start`, `y3_value`, `copm_diff`, `brl_diff`, `kes_diff`, `eur_diff`, `x_d`, `vix_avg`, `oil_return`, `us_cpi_surprise`, `banrep_rate_surprise`, `fed_funds_weekly`, `intervention_dummy`); `source_methodology` resolves from `_audit_summary.json` to `y3_v2_co_dane_br_bcb_eu_eurostat_ke_skip_3country_ke_unavailable` (Rev-5.3.2 default-flip post-Task 11.O Step-0). Every assertion in this cell encodes a load-bearing acceptance criterion: any future drift to row count, window, column set, methodology tag, or Friday-anchor invariant surfaces here rather than propagating silently into the §1 OLS coefficient.

**Seed pinned via `env.pin_seed(seed=42)`.** The `random` and `numpy.random` global RNG state is now deterministic in this process; `PYTHONHASHSEED=42` propagates to nbconvert child processes only (per the `env.py` docstring caveat — intended behavior). NB2 §1 is closed-form OLS + HAC(4) Newey–West and is therefore deterministic by construction; the seed pin matters for NB3 bootstrap re-confirmation cells, but is established here so every downstream cell in the notebook inherits the same RNG state without re-pinning.

**Scope framing under Rev-5.3.5 (Minteo-fintech scope-mismatch close-out).** This panel was constructed against the X_d source address `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606`; per the Rev-5.3.5 β-disposition memo at `contracts/.scratch/2026-04-26-mr-beta-1-1-halt-resolution-beta.md`, that address is reclassified as Minteo-fintech (out of Mento-native scope per `project_abrigo_mento_native_only`). The canonical Mento-native `StableTokenCOP` address is `0x8A567e2aE79CA692Bd748aB832081C45de4041eA`; estimation against the Mento-native COPm hedge-demand surface dispatches separately under Task 11.P β-track (Task 11.P.spec-β / Task 11.P.exec-β). NB2 reproduces the Rev-2 published Minteo-fintech X_d estimates byte-exact — so Rev-2 closes scope-mismatch close-out on the Minteo-fintech X_d, not as a test of the Mento-native COPm hedge-demand surface.

**Forward pointer.** NB2 §1 (primary OLS+HAC(4) Newey–West regression on the spec-§4.1 equation `y3_value ~ x_d + vix_avg + oil_return + us_cpi_surprise + banrep_rate_surprise + fed_funds_weekly + intervention_dummy`) follows in NB-α sub-task 9, dispatched separately per the trio-checkpoint discipline. The published reproduction target (β̂_X_d = −2.7987e−8, HAC(4) SE = 1.4234e−8, t-stat = −1.9662, n = 76, gate verdict FAIL) is byte-exact-immutable.


---

## §1 — Primary OLS + HAC(4) Newey–West regression (gate-bearing row)

### Why-markdown (4-part citation block)

**Reference.**

- Newey, W. K. and West, K. D. (1987), *A Simple, Positive Semi-Definite, Heteroskedasticity and Autocorrelation Consistent Covariance Matrix*, Econometrica 55(3), 703–708 — `\cite{neweyWest1987simple}` in `references.bib`. The HAC(4) Newey–West covariance estimator with bandwidth `maxlags=4` is the pre-registered standard-error layer for the primary regression.
- Rev-2 spec autonomous track A at `contracts/.scratch/2026-04-25-task110-rev2-spec-A-autonomous.md` §3 (functional form `Y₃ = α + β · X_d + γ' · controls + ε`; 6 controls = `vix_avg + oil_return + us_cpi_surprise + banrep_rate_surprise + fed_funds_weekly + intervention_dummy` per NB1 §4); §6 (T3b 90% one-sided gate definition: PASS iff `β̂ − 1.28 · SE > 0`); §11.A (mean-β identification is necessary-but-insufficient for convex-payoff pricing — the T3b verdict bounds linear-payoff hedge calibration only; convex-payoff fitness is Rev-3 ζ-group, deferred).
- Project memory `project_mdes_formulation_pin` (Cohen f² formulation pin; sha256 = `4940360dcd2987…cefa`; pinned source at `contracts/scripts/carbon_calibration.py :: required_power(n_obs, k_regressors, mdes_sd, alpha=0.10, df1=6)`; free-tuning `MDES_SD` upward to chase a target power figure is anti-fishing-banned).
- Phase 5b published gate verdict at `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json` (canonical immutable Rev-5.3.2 Row-1 values: `β̂_X_d = -2.7987050503705652e-08`, `HAC(4) SE = 1.4234226026833985e-08`, `t-stat = -1.9661799981920483`, two-sided p = `0.04927782209941108`, one-sided 90% lower = `-4.6206859818053154e-08`, n = 76, gate verdict FAIL).
- Phase 5b 14-row resolution matrix at `contracts/.scratch/2026-04-25-task110-rev2-analysis/estimates.md` (Row 1 is the gate-bearing primary; Rows 2–14 follow in NB2 §2 onward).
- Rev-5.3.5 β-disposition memo at `contracts/.scratch/2026-04-26-mr-beta-1-1-halt-resolution-beta.md` (the disposition that reclassifies the X_d source address `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606` as Minteo-fintech, OUT of Mento-native scope; Rev-2 closes scope-mismatch close-out on the Minteo-fintech X_d, not as a test of the Mento-native COPm hedge-demand surface at `0x8A567e2aE79CA692Bd748aB832081C45de4041eA`).

**Why used.** OLS + HAC(4) Newey–West is the pre-committed primary estimator per Rev-2 spec §3; HAC(4) accounts for serial correlation up to lag 4 (the Newey–West bandwidth pre-registered in the spec) without imposing a parametric AR/MA structure on the residuals — this matches the standard small-sample weekly-frequency convention for n ≈ 76. The one-sided T3b 90% lower bound `β̂ − 1.28 · SE` is the gate-bearing quantity per §6 pre-registration — **NOT** a two-sided 90% Wald CI; the one-sided form is required because the directional alternative (positive β: rich-asset returns / working-class CPI differential rises with carbon-basket user-volume X_d) was pre-committed before estimation, and a two-sided form would over-cover the null. The byte-exact reproduction of the published Row-1 estimates against the canonical immutable `gate_verdict.json` is the load-bearing acceptance criterion: it validates the panel-load + estimator chain end-to-end (NB1 panel construction → NB2 §0 panel-load contract → NB2 §1 OLS + HAC(4)) without re-introducing analytical drift across the migration. Statistical power at the primary-row n is computed via the pinned Cohen f² formulation (sha256-tamper-evident); free-tuning `MDES_SD` upward is anti-fishing-banned.

**Relevance to results.** The gate verdict (FAIL) is the load-bearing analytical output of NB2: under the pre-registered T3b 90% one-sided rule, `β̂ − 1.28 · SE = -4.6207e-8 < 0`, so the directional null cannot be rejected. Under Rev-5.3.5 β-disposition, this FAIL is the **Minteo-fintech scope-mismatch close-out** — Rev-2 measured X_d on the Minteo-fintech address `0xc92e8fc2…` which is OUT of Mento-native scope per `project_abrigo_mento_native_only`; Rev-2 closes scope-mismatch close-out on the Minteo-fintech X_d, NOT a test of the Mento-native COPm hedge-demand surface. The Mento-native COPm hedge-demand-surface evaluation is β-track Rev-3 (deferred to Task 11.P.spec-β / Task 11.P.exec-β). The published Row-1 power 0.8689 ≥ POWER_MIN = 0.80 confirms the primary row is structurally sound under the pre-registered Cohen f² discipline (the FAIL is not a power-deficit artifact). NB2 §1 emits `notebooks/abrigo_y3_x_d/estimates/primary_estimate.json` with the byte-exact Row-1 record (β̂, SE, t-stat, p-values, lower-90, n, window, gate verdict, power, MDES_FORMULATION_HASH, decision_hash) for downstream NB3 consumption.

**Connection to product.** The Abrigo simulator's mean-β / linear-hedge calibration consumes the published Row-1 β̂ and HAC(4) SE as the first-stage payoff-matching anchor for the convex (option-like) instruments Abrigo sells against macroeconomic shocks viewed through the inequality lens (`project_abrigo_convex_instruments_inequality`). Under the FAIL gate, the linear-hedge calibration on the Minteo-fintech X_d source is bounded — the simulator should not interpret the Minteo-fintech β̂ as a Mento-native COPm hedge-demand surface. Per Rev-2 spec §11.A, mean-β alone is insufficient for convex-payoff pricing; convex fitness lives in Rev-3 ζ-group (quantile / GARCH-X / lower-tail / option-implied vol), held under Task 11.O.ζ-α per `project_compact_survival_2026_04_26`. The Mento-native COPm-surface estimation runs through this same notebook scaffolding pattern under β-track.



In [2]:
"""NB2 §1 — primary OLS + HAC(4) Newey–West regression on the Rev-2 panel.

Reproduces the Phase 5b published Row-1 byte-exact (within 1e-12 floating-point
tolerance) against `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json` and
emits `primary_estimate.json` for downstream NB3 consumption. Functional-Python
style: frozen dataclasses, free pure functions, full typing, no inheritance.
"""
from __future__ import annotations

import hashlib
import importlib.util
import inspect
import json
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Final, Mapping

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ---- Resolve `carbon_calibration.required_power` (the pinned Cohen f² source) ----
#
# `contracts/scripts/` is not on `sys.path` from the notebook cwd; load by file
# path the same way `env.py` is loaded in §0. We hold the loaded module under
# a private name so the SHA256 verification reads the *exact* in-process source.

def _locate_carbon_calibration() -> Path:
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        candidate = parent / "contracts" / "scripts" / "carbon_calibration.py"
        if candidate.is_file():
            return candidate
        candidate2 = parent / "scripts" / "carbon_calibration.py"
        if candidate2.is_file():
            return candidate2
    raise RuntimeError(
        f"Could not locate contracts/scripts/carbon_calibration.py from cwd={cwd}"
    )


_CARBON_CALIB_PATH: Final[Path] = _locate_carbon_calibration()
_carbon_spec = importlib.util.spec_from_file_location(
    "abrigo_carbon_calibration", _CARBON_CALIB_PATH
)
carbon_calibration = importlib.util.module_from_spec(_carbon_spec)
sys.modules["abrigo_carbon_calibration"] = carbon_calibration
_carbon_spec.loader.exec_module(carbon_calibration)

# ---- Pre-committed Rev-2 spec constants (binding pre-registration) ----

CONTROL_COLUMNS: Final[tuple[str, ...]] = (
    "vix_avg",
    "oil_return",
    "us_cpi_surprise",
    "banrep_rate_surprise",
    "fed_funds_weekly",
    "intervention_dummy",
)
"""Six controls per Rev-2 spec §3 + NB1 §4 functional-form pre-registration."""

T3B_Z90: Final[float] = 1.28
"""Standard normal one-sided 90% critical value per Rev-2 spec §6 gate definition."""

MDES_SD: Final[float] = 0.40
"""Pre-registered MDES per `project_mdes_formulation_pin`."""

K_REGRESSORS_FOR_POWER: Final[int] = 13
"""Total panel-column count k for Cohen f² power per spec §6 (NOT design-matrix width)."""

PRIMARY_REV532_DECISION_HASH: Final[str] = (
    "6a5f9d1b05c18defd8b30c4b3cef6af896d6e45a2a26c1c60aa342da0a5a443c"
)
"""Rev-4 decision_hash carried forward into Rev-5.3.x per the migration contract."""

EXPECTED_PUBLISHED: Final[Mapping[str, float | int]] = {
    "beta_x_d": -2.7987050503705652e-08,
    "se": 1.4234226026833985e-08,
    "t_stat": -1.9661799981920483,
    "p_two_sided": 0.04927782209941108,
    "lower_90_one_sided": -4.6206859818053154e-08,
    "n": 76,
}
"""Byte-exact published Row-1 values from `gate_verdict.json` (canonical immutable)."""


@dataclass(frozen=True, slots=True)
class PrimaryEstimate:
    """Frozen-dataclass record of the gate-bearing Row-1 OLS + HAC(4) result.

    Emitted byte-exact to `primary_estimate.json` for downstream NB3 consumption.
    """

    beta_x_d: float
    se_hac4: float
    t_stat: float
    p_two_sided: float
    p_one_sided: float
    lower_90_one_sided: float
    n: int
    k_regressors_design: int
    k_regressors_for_power: int
    window_start: str
    window_end: str
    estimator: str
    hac_bandwidth: int
    z90_one_sided: float
    gate_verdict: str
    gate_passes: bool
    power: float
    mdes_sd: float
    mdes_formulation_hash: str
    decision_hash: str
    source_methodology: str


# ---- Pure functions (no inheritance, no globals mutated) ----

def _verify_mdes_formulation_hash() -> str:
    """Recompute SHA256 of `required_power` source and assert byte-exact pin match.

    Free-tuning is anti-fishing-banned per `project_mdes_formulation_pin`; this
    guard fires loudly if the pinned source has drifted.
    """
    src = inspect.getsource(carbon_calibration.required_power)
    live_sha = hashlib.sha256(src.encode("utf-8")).hexdigest()
    if live_sha != carbon_calibration.MDES_FORMULATION_HASH:
        raise RuntimeError(
            "MDES formulation hash drift: live SHA256 of `required_power` "
            f"= {live_sha}; pinned = {carbon_calibration.MDES_FORMULATION_HASH}. "
            "Re-pinning requires (a) design-doc revision, (b) CORRECTIONS block, "
            "(c) full 3-way review per anti-fishing protocol."
        )
    return live_sha


def build_design_matrix(panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Build the design matrix X = [intercept, x_d, 6 controls] and outcome y.

    Pure function; does not mutate `panel`. Column order is intercept-first
    so `statsmodels` reports the intercept on the leading row and `x_d` next.
    """
    X = panel[["x_d", *CONTROL_COLUMNS]].astype(float).copy()
    X = sm.add_constant(X, prepend=True)  # intercept first
    y = panel["y3_value"].astype(float).copy()
    return X, y


def fit_ols_hac4(X: pd.DataFrame, y: pd.Series) -> sm.regression.linear_model.RegressionResultsWrapper:
    """Fit OLS with Newey–West HAC(4) covariance per Rev-2 spec §3."""
    return sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})


def compute_primary_estimate(
    panel: pd.DataFrame,
    *,
    methodology: str,
    z90: float = T3B_Z90,
    mdes_sd: float = MDES_SD,
    k_regressors_for_power: int = K_REGRESSORS_FOR_POWER,
) -> PrimaryEstimate:
    """End-to-end primary-row estimator. Closed-form; deterministic by construction."""
    X, y = build_design_matrix(panel)
    res = fit_ols_hac4(X, y)

    beta = float(res.params["x_d"])
    se = float(res.bse["x_d"])
    t = float(res.tvalues["x_d"])
    p_two = float(res.pvalues["x_d"])
    # One-sided right-tail p (HA: β > 0). For β̂ < 0 this is > 0.5; for β̂ > 0 it is < 0.5.
    # We compute it from the t-stat sign symmetry: p_one = 1 - Φ(t) under large-T HAC.
    # Using the same ttest_1samp-style convention as Phase 5b: p_one = (1 - p_two/2) when t<0,
    # else (p_two/2) when t>0. Equivalent to: p_one = 1 - p_two/2 for t<0, p_two/2 for t>0.
    p_one = 1.0 - p_two / 2.0 if beta < 0 else p_two / 2.0

    lower_90 = beta - z90 * se
    gate_passes = lower_90 > 0.0
    gate_verdict = "PASS" if gate_passes else "FAIL"

    n = int(len(panel))
    power = float(carbon_calibration.required_power(n, k_regressors_for_power, mdes_sd))
    mdes_hash = _verify_mdes_formulation_hash()

    dt_min = panel["week_start"].min().strftime("%Y-%m-%d")
    dt_max = panel["week_start"].max().strftime("%Y-%m-%d")

    return PrimaryEstimate(
        beta_x_d=beta,
        se_hac4=se,
        t_stat=t,
        p_two_sided=p_two,
        p_one_sided=p_one,
        lower_90_one_sided=lower_90,
        n=n,
        k_regressors_design=int(X.shape[1]),
        k_regressors_for_power=k_regressors_for_power,
        window_start=dt_min,
        window_end=dt_max,
        estimator="ols_hac4",
        hac_bandwidth=4,
        z90_one_sided=z90,
        gate_verdict=gate_verdict,
        gate_passes=gate_passes,
        power=power,
        mdes_sd=mdes_sd,
        mdes_formulation_hash=mdes_hash,
        decision_hash=PRIMARY_REV532_DECISION_HASH,
        source_methodology=methodology,
    )


def assert_byte_exact(actual: PrimaryEstimate, expected: Mapping[str, float | int]) -> None:
    """Byte-exact (within 1e-12) reproduction guard against the published Row-1.

    The 5 canonical published values must match `gate_verdict.json` byte-exact;
    n must match exactly. Any drift HALTs.
    """
    tolerance = 1e-12
    pairs = (
        ("beta_x_d", actual.beta_x_d),
        ("se", actual.se_hac4),
        ("t_stat", actual.t_stat),
        ("p_two_sided", actual.p_two_sided),
        ("lower_90_one_sided", actual.lower_90_one_sided),
    )
    for key, value in pairs:
        diff = abs(value - expected[key])
        if diff >= tolerance:
            raise AssertionError(
                f"Byte-exact reproduction drift on {key}: got {value!r}, "
                f"expected {expected[key]!r}, |diff| = {diff:.2e} "
                f"(tolerance = {tolerance:.0e})"
            )
    if actual.n != expected["n"]:
        raise AssertionError(
            f"n drift: got {actual.n}, expected {expected['n']}"
        )


def emit_primary_estimate_json(estimate: PrimaryEstimate, target_path: Path) -> None:
    """Write `primary_estimate.json` with deterministic key ordering."""
    target_path.parent.mkdir(parents=True, exist_ok=True)
    with target_path.open("w", encoding="utf-8") as fh:
        json.dump(asdict(estimate), fh, indent=2, sort_keys=True)
        fh.write("\n")


# ---- Execute the §1 primary regression ----

# Reuse `panel_primary` and `contract` from §0; methodology is the audited tag.
estimate = compute_primary_estimate(
    panel_primary,
    methodology=contract.source_methodology,
)

# Byte-exact reproduction guard against the canonical immutable gate verdict.
assert_byte_exact(estimate, EXPECTED_PUBLISHED)

# Power assertion: 0.8689 ≥ POWER_MIN = 0.80 per Rev-5.3.1 anti-fishing invariant.
assert estimate.power >= 0.80, (
    f"Power deficit: got {estimate.power!r}, expected >= 0.80 "
    f"(POWER_MIN per project_mdes_formulation_pin)"
)

# Gate verdict: T3b PASS iff `β̂ − 1.28 · SE > 0`. Expected = FAIL.
assert estimate.gate_verdict == "FAIL", (
    f"Gate verdict drift: got {estimate.gate_verdict!r}, expected 'FAIL' "
    f"(β̂ = {estimate.beta_x_d:.4e}, SE = {estimate.se_hac4:.4e}, "
    f"lower_90 = {estimate.lower_90_one_sided:.4e})"
)

# Emit `primary_estimate.json` to the canonical estimates/ directory (sibling to gate_verdict.json).
_estimates_dir = _ABRIGO_DIR / "estimates"
_primary_estimate_path = _estimates_dir / "primary_estimate.json"
emit_primary_estimate_json(estimate, _primary_estimate_path)

# Sanity print: round-tripped record + reproduction-diff snapshot.
print("§1 OLS + HAC(4) Newey–West — primary row (gate-bearing)")
print("=" * 64)
print(f"β̂_X_d                      : {estimate.beta_x_d!r}")
print(f"HAC(4) SE                   : {estimate.se_hac4!r}")
print(f"t-stat                      : {estimate.t_stat!r}")
print(f"two-sided p (T3a)           : {estimate.p_two_sided!r}")
print(f"one-sided p (right-tail)    : {estimate.p_one_sided!r}")
print(f"one-sided 90% lower bound   : {estimate.lower_90_one_sided!r}")
print(f"n                           : {estimate.n}")
print(f"window                      : [{estimate.window_start}, {estimate.window_end}]")
print(f"design-matrix width (k+1)   : {estimate.k_regressors_design}")
print(f"k for Cohen f² power        : {estimate.k_regressors_for_power}")
print(f"power(n={estimate.n}, k={estimate.k_regressors_for_power}, MDES_SD={estimate.mdes_sd}) : {estimate.power!r}")
print(f"MDES_FORMULATION_HASH       : {estimate.mdes_formulation_hash}")
print(f"decision_hash               : {estimate.decision_hash}")
print(f"gate_verdict (T3b one-sided): {estimate.gate_verdict}")
print()
print(f"primary_estimate.json -> {_primary_estimate_path}")
print()
print("Byte-exact reproduction (|diff| vs gate_verdict.json):")
for key in ("beta_x_d", "se", "t_stat", "p_two_sided", "lower_90_one_sided"):
    actual_val = {
        "beta_x_d": estimate.beta_x_d,
        "se": estimate.se_hac4,
        "t_stat": estimate.t_stat,
        "p_two_sided": estimate.p_two_sided,
        "lower_90_one_sided": estimate.lower_90_one_sided,
    }[key]
    diff = abs(actual_val - EXPECTED_PUBLISHED[key])
    print(f"  {key:<22}: |diff| = {diff:.2e}")


§1 OLS + HAC(4) Newey–West — primary row (gate-bearing)
β̂_X_d                      : -2.7987050503705652e-08
HAC(4) SE                   : 1.4234226026833985e-08
t-stat                      : -1.9661799981920483
two-sided p (T3a)           : 0.04927782209941108
one-sided p (right-tail)    : 0.9753610889502945
one-sided 90% lower bound   : -4.6206859818053154e-08
n                           : 76
window                      : [2024-09-27, 2026-03-13]
design-matrix width (k+1)   : 8
k for Cohen f² power        : 13
power(n=76, k=13, MDES_SD=0.4) : 0.8689122648203158
MDES_FORMULATION_HASH       : 4940360dcd298738a1f7321c1573bc3aad01b8a4c5acbc546d0855276389cefa
decision_hash               : 6a5f9d1b05c18defd8b30c4b3cef6af896d6e45a2a26c1c60aa342da0a5a443c
gate_verdict (T3b one-sided): FAIL

primary_estimate.json -> /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/notebooks/abrigo_y3_x_d/estimates/primary_estimate.json

Byte-exact reproduction (|diff| vs gat

### Interpretation

**Byte-exact reproduction PASSES against the canonical immutable gate verdict.** All five published Row-1 values match `notebooks/abrigo_y3_x_d/estimates/gate_verdict.json` to within the 1e-12 floating-point tolerance (in fact, `|diff| = 0.00e+00` in every component): `β̂_X_d = -2.7987050503705652e-08`, `HAC(4) SE = 1.4234226026833985e-08`, `t-stat = -1.9661799981920483`, two-sided `p = 0.04927782209941108`, one-sided 90% lower bound `= -4.6206859818053154e-08`, `n = 76`, window = `[2024-09-27, 2026-03-13]`. The OLS + Newey–West estimator chain (panel-load via NB2 §0 → design matrix `[intercept, x_d, vix_avg, oil_return, us_cpi_surprise, banrep_rate_surprise, fed_funds_weekly, intervention_dummy]` → `statsmodels.api.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})`) reproduces the Phase 5b published primary row without analytical drift across the migration. The `MDES_FORMULATION_HASH` recompute against `inspect.getsource(carbon_calibration.required_power)` returns the pinned `4940360dcd2987…cefa` byte-exact, validating the anti-fishing tamper-evidence layer at notebook-execute time.

**Gate verdict T3b = FAIL.** Under the pre-registered T3b 90% one-sided rule (`β̂ − 1.28 · SE > 0` ⇒ PASS, else FAIL), the lower-90% bound `= -4.6207e-8 < 0` falls below the directional zero, so the null `β ≤ 0` cannot be rejected. The two-sided T3a p-value of 0.0493 is technically significant at the 5% level but signed *against* the directional alternative — i.e., the point estimate is negative when the pre-committed alternative was positive. This is the canonical T3b discipline: the one-sided gate enforces directional pre-commitment and prevents reversing the sign of the alternative after seeing the data. Statistical power at the primary-row n is `power(76, k=13, MDES_SD=0.40) = 0.8689` per the pinned Cohen f² formulation — exceeds `POWER_MIN = 0.80` and therefore the FAIL is a **structural** outcome on the Minteo-fintech X_d series, not a power-deficit artifact.

**Scope framing under Rev-5.3.5 (Minteo-fintech scope-mismatch close-out).** The X_d source address used in this Rev-2 panel is `0xc92e8fc2947e32f2b574cca9f2f12097a71d5606`, which the Rev-5.3.5 β-disposition reclassifies as Minteo-fintech (out of Mento-native scope per `project_abrigo_mento_native_only`). Rev-2 closes scope-mismatch close-out on the Minteo-fintech X_d — the FAIL is a Minteo-fintech scope-mismatch close-out, **not** a test of the Mento-native COPm hedge-demand surface at `0x8A567e2aE79CA692Bd748aB832081C45de4041eA`. Per Rev-2 spec §11.A, the mean-β / linear-hedge calibration this regression produces is necessary-but-insufficient for the convex (option-like) instruments Abrigo sells against macroeconomic shocks viewed through the inequality lens (`project_abrigo_convex_instruments_inequality`); convex-payoff fitness lives in Rev-3 ζ-group (quantile / GARCH-X / lower-tail / option-implied vol), held under Task 11.O.ζ-α per `project_compact_survival_2026_04_26`. The Mento-native COPm-surface test is β-track Rev-3, dispatched separately under Task 11.P.spec-β / Task 11.P.exec-β.

**Emitted artifact.** `notebooks/abrigo_y3_x_d/estimates/primary_estimate.json` carries the byte-exact Row-1 record (β̂, SE, t-stat, p-values, lower-90, n, window, gate verdict, gate_passes flag, power, MDES_SD, MDES_FORMULATION_HASH, decision_hash, source_methodology) under deterministic key ordering (`json.dumps(..., indent=2, sort_keys=True)`) for downstream NB3 consumption.

**Forward pointer.** NB2 §2 (sensitivity Row 2 — Politis–Romano stationary-block bootstrap reconciliation, mean block length = 4, 10 000 resamples, empirical 5/95-quantile CI) follows in NB-α sub-task 10, dispatched separately per the trio-checkpoint discipline. Per Phase 5b, Row 2 reconciles with Row 1 at the 50% containment level (`agrees_with_hac_at_50pct_containment = true`); the bootstrap SE = 2.222e−8 is wider than HAC(4) but the empirical right-tail p does not move the gate verdict.
